# 02 — Vanilla Diffusion Policy Baseline

분리된 구조 (`src/*.py`) 기반의 깨끗한 Step 2 노트북.

**선행 조건**:
- `01_data_pipeline.ipynb` 실행 완료 → `demos_ant_planC.npz`, `norm_stats.npz` 가 Drive에 있음
- `src/dataset.py`, `src/models.py`, `src/training.py`, `src/sampling.py` 가 Drive에 있음

**이 노트북에서 하는 것**:
1. Drive mount + `src/` import + 데이터 로드
2. Conditional 1D U-Net 빌드 + sanity forward
3. DDPM training (또는 기존 ckpt 로드)
4. DDIM sampling sanity (offline)
5. Ant 환경 rollout sanity

**제안서 §6.3 baseline의 두 번째 항목** — 이 모델이 작동해야 phase-conditioned 비교 baseline이 됨.

## 1. Setup — Drive mount + src import

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass

import os, sys
from pathlib import Path

REPO_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next((p for p in REPO_ROOT_CANDIDATES if (p / 'src' / 'paths.py').exists()), None)
assert REPO_ROOT is not None, 'repo root with src/paths.py not found; run this notebook from the cloned repository'
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from paths import (
    ARTIFACT_ROOT, DATA_DIR, CHECKPOINTS_DIR, RESULTS_DIR,
    FIGURES_DIR, VIDEOS_DIR, ensure_artifact_dirs,
)
ensure_artifact_dirs()

print(f'✓ src 경로 등록: {SRC_DIR}')
print(f'✓ artifact root: {ARTIFACT_ROOT}')
print(f'  src 파일: {[f.name for f in SRC_DIR.glob("*.py")]}')


## 2. diffusers 설치 + import

런타임을 새로 시작했다면 diffusers가 없을 수 있음. 이미 있으면 빠르게 통과.

In [ ]:
!pip install -q -r {REPO_ROOT / 'requirements.txt'}
print("✓ requirements.txt 기반 의존성 준비 완료")

In [ ]:
import math
import time
import shutil
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# src 모듈
from configs import get_experiment_config
from dataset  import load_project_data, build_loaders
from models   import count_params
from training import train_diffusion_policy, save_checkpoint, load_checkpoint
from sampling import (
    sample_action_chunk, rollout, rollout_multi_seed,
    diagnose_obs_distribution,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = get_experiment_config('vanilla')
train_cond_fn = cfg.resolve_train_cond_fn()
sample_cond_fn = cfg.resolve_sample_cond_fn()

# 기존 diagnostic/evaluation cell 호환용 alias. 실제 값은 cfg에서 resolve됨.
vanilla_sample_cond_fn = sample_cond_fn

print(f"PyTorch {torch.__version__}, device={device}")
print(f"Experiment config: {cfg.name} — {cfg.display_name}")


## 3. 데이터 로드 + DataLoader 구성

이 두 줄로 Step 1 산출물을 다 가져옴.

In [ ]:
data = load_project_data(DATA_DIR)

print(f"OBS_DIM={data['OBS_DIM']}, ACT_DIM={data['ACT_DIM']}")
print(f"obs_horizon={data['OBS_HORIZON']}, pred_horizon={data['PRED_HORIZON']}, "
      f"action_horizon={data['ACTION_HORIZON']}")
print(f"Train: {len(data['train_eps'])} eps | Val: {len(data['val_eps'])} eps")
print(f"Freq window: {data['freq_window_mean']:.3f} ± {data['freq_window_std']:.3f} Hz")

# Reproducibility
SEED = data['seed']
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

train_ds, val_ds, train_loader, val_loader = build_loaders(
    data, batch_size=cfg.data.batch_size, num_workers=cfg.data.num_workers,
)


## 4. 모델 빌드 + sanity forward

In [ ]:
model = cfg.build_model(data, device=device)
n = count_params(model)
print(f"Total params:     {n['total']/1e6:.2f}M")
print(f"Trainable params: {n['trainable']/1e6:.2f}M")

# Sanity forward
B = 4
fake_action = torch.randn(B, data['PRED_HORIZON'], data['ACT_DIM'], device=device)
fake_t = torch.randint(0, cfg.diffusion.num_train_timesteps, (B,), device=device)
fake_cond = torch.randn(B, data['OBS_HORIZON'] * data['OBS_DIM'], device=device)
with torch.no_grad():
    out = model(fake_action, fake_t, fake_cond)
assert out.shape == fake_action.shape
print(f"\n✓ Forward pass: in {tuple(fake_action.shape)} → out {tuple(out.shape)}")


## 5. Noise scheduler (DDPM)

In [ ]:
NUM_TRAIN_TIMESTEPS = cfg.diffusion.num_train_timesteps
NUM_INFERENCE_STEPS  = cfg.diffusion.num_inference_steps

noise_scheduler = cfg.build_noise_scheduler()
print(f"DDPM: {NUM_TRAIN_TIMESTEPS} train steps, "
      f"{noise_scheduler.config.beta_schedule}, "
      f"{noise_scheduler.config.prediction_type}-prediction")

# Sampling 함수에 넘길 dict (DDIM scheduler가 같은 config로 만들어짐)
ns_config = cfg.noise_scheduler_config()


## 6. 학습 또는 기존 ckpt 로드

`TRAIN=True`면 학습 실행 (T4 기준 100 epoch ≈ 15–25분).
`TRAIN=False`면 Drive의 기존 ckpt 로드 (학습 다시 안 돌림).

### 분리 구조 검증 시나리오

이미 합쳐진 노트북에서 학습한 ckpt가 있다면:
1. 그 ckpt를 `CHECKPOINTS_DIR / 'vanilla_dp_ckpt.pt'` 경로로 저장 (이미 그 경로일 것)
2. 여기서 `TRAIN=False`로 두고 실행
3. Sampling/rollout 셀이 정상 작동하면 → src 분리 성공

In [ ]:
TRAIN = False   # 학습된 ckpt가 이미 있으면 False
CKPT_PATH = cfg.checkpoint_path(CHECKPOINTS_DIR)

ema = cfg.build_ema(model)

if TRAIN:
    train_losses, val_log, best_ema_state = train_diffusion_policy(
        model, ema, noise_scheduler,
        train_loader, val_loader,
        cond_fn=train_cond_fn,
        device=device,
        **cfg.training_kwargs(),
    )
    checkpoint_config = cfg.to_dict()
    checkpoint_config['data_metadata'] = {
        'OBS_DIM': data['OBS_DIM'], 'ACT_DIM': data['ACT_DIM'],
        'OBS_HORIZON': data['OBS_HORIZON'],
        'PRED_HORIZON': data['PRED_HORIZON'],
        'ACTION_HORIZON': data['ACTION_HORIZON'],
    }
    save_checkpoint(
        CKPT_PATH, model, ema, train_losses, val_log, best_ema_state,
        config=checkpoint_config,
    )
else:
    meta = load_checkpoint(CKPT_PATH, model, ema, device=device)
    train_losses    = meta['train_losses']
    val_log         = meta['val_log']
    best_ema_state  = meta['best_ema_state']

USE_BEST_EMA = True

if USE_BEST_EMA and best_ema_state is not None:
    # ema의 state_dict 형태를 유지하면서 best 값들로 덮어쓰기
    cur = ema.state_dict()
    cur.update(best_ema_state)
    ema.load_state_dict(cur)
    print(f"✓ Best EMA 적용")
elif best_ema_state is None:
    print("⚠ best_ema_state 없음 — Final EMA 사용")
else:
    print("Final EMA 사용")


## 7. Loss curve

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.plot(np.arange(1, len(train_losses) + 1), train_losses, label='train', alpha=0.8)
if val_log:
    val_epochs = [v[0] for v in val_log]
    val_vals   = [v[1] for v in val_log]
    ax.plot(val_epochs, val_vals, 'o-', label='val (EMA)', color='red')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE loss')
ax.set_title('Vanilla DP — Training Curves')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_yscale('log')
plt.tight_layout()
out_png = FIGURES_DIR / 'vanilla_dp_loss.png'
plt.savefig(out_png, dpi=80, bbox_inches='tight')
plt.show()
print(f"✓ {out_png}")

## 8. Offline sampling sanity

Val set의 obs window 64개로 chunk 샘플링 → 분포가 training과 비슷한지 확인.

In [ ]:
# Val set에서 64개 batch 만들기
val_obs_list, val_act_true_list = [], []
for i in range(min(64, len(val_ds))):
    s = val_ds[i]
    val_obs_list.append(s['obs'])
    val_act_true_list.append(s['action'])
val_obs_batch = torch.stack(val_obs_list)
val_act_true  = torch.stack(val_act_true_list).numpy()

print(f"Sampling {len(val_obs_batch)} chunks (DDIM {NUM_INFERENCE_STEPS} steps)...")
t0 = time.time()
sampled = sample_action_chunk(
    model, ema, ns_config,
    obs_window=val_obs_batch,
    cond_fn=vanilla_sample_cond_fn,
    pred_horizon=data['PRED_HORIZON'],
    act_dim=data['ACT_DIM'],
    num_inference_steps=NUM_INFERENCE_STEPS,
    device=device,
).numpy()
print(f"  {time.time()-t0:.1f}s")

# 차원별 분포 비교
print(f"\n{'dim':>4} | {'true mean':>11} | {'sampled mean':>13} | "
      f"{'true std':>10} | {'sampled std':>12}")
for d in range(data['ACT_DIM']):
    print(f"{d:>4} | {val_act_true[:,:,d].mean():>11.3f} | "
          f"{sampled[:,:,d].mean():>13.3f} | "
          f"{val_act_true[:,:,d].std():>10.3f} | "
          f"{sampled[:,:,d].std():>12.3f}")

In [ ]:
# Histogram 시각화
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for d in range(data['ACT_DIM']):
    ax = axes.flat[d]
    ax.hist(val_act_true[:,:,d].flatten(), bins=30, alpha=0.5,
            label='true', color='blue', density=True)
    ax.hist(sampled[:,:,d].flatten(), bins=30, alpha=0.5,
            label='sampled', color='red', density=True)
    ax.set_title(f'action dim {d}')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.suptitle('Vanilla DP — Sampled vs True (정규화)')
plt.tight_layout()
out_png = FIGURES_DIR / 'vanilla_dp_action_dist.png'
plt.savefig(out_png, dpi=80, bbox_inches='tight')
plt.show()
print(f"✓ {out_png}")

## 9. Chunk 시각화 — 같은 obs로 4번 샘플링

Diffusion stochasticity 확인. 약간 다르지만 비슷한 shape이어야 함.

In [ ]:
ep_obs = val_obs_batch[0:1]
true_chunk = val_act_true[0]

samples = []
for _ in range(4):
    s = sample_action_chunk(
        model, ema, ns_config,
        obs_window=ep_obs,
        cond_fn=vanilla_sample_cond_fn,
        pred_horizon=data['PRED_HORIZON'],
        act_dim=data['ACT_DIM'],
        num_inference_steps=NUM_INFERENCE_STEPS,
        device=device,
    ).numpy()[0]
    samples.append(s)

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for d in range(data['ACT_DIM']):
    ax = axes.flat[d]
    ax.plot(true_chunk[:, d], 'k-', linewidth=2, label='true', alpha=0.8)
    for i, s in enumerate(samples):
        ax.plot(s[:, d], '--', alpha=0.5, label=f'sample {i+1}' if d == 0 else None)
    ax.axhline( 1, color='r', ls=':', alpha=0.3)
    ax.axhline(-1, color='r', ls=':', alpha=0.3)
    ax.set_title(f'action dim {d}')
    ax.set_xlabel('step within chunk')
    ax.grid(True, alpha=0.3)
    if d == 0:
        ax.legend(fontsize=7, ncol=2)
plt.suptitle('Vanilla DP — True vs 4 Samples (same obs)')
plt.tight_layout()
out_png = FIGURES_DIR / 'vanilla_dp_chunk_compare.png'
plt.savefig(out_png, dpi=80, bbox_inches='tight')
plt.show()
print(f"✓ {out_png}")

## 10. MuJoCo Ant rollout sanity

자빠지지 않고 100+ step 가면 Vanilla DP baseline 작동 확인.

In [ ]:
import gymnasium as gym

env = gym.make('Ant-v5')
test_obs, _ = env.reset(seed=SEED)
print(f"Ant-v5 default obs dim: {test_obs.shape[0]}")
print(f"학습 OBS_DIM={data['OBS_DIM']} → "
      f"{'일치' if test_obs.shape[0] == data['OBS_DIM'] else '불일치 (truncate 사용)'}")

In [ ]:
print("=== Vanilla DP Rollout (5 seeds) ===\n")
results = rollout_multi_seed(
    model, ema, env, n_seeds=5,
    noise_scheduler_config=ns_config,
    obs_mean=data['obs_mean'], obs_std=data['obs_std'],
    act_min=data['act_min'], act_range=data['act_range'],
    cond_fn=vanilla_sample_cond_fn,
    obs_horizon=data['OBS_HORIZON'],
    pred_horizon=data['PRED_HORIZON'],
    action_horizon=data['ACTION_HORIZON'],
    obs_dim=data['OBS_DIM'],
    act_dim=data['ACT_DIM'],
    num_inference_steps=NUM_INFERENCE_STEPS,
    max_steps=300,
    device=device,
)

surv_mean = np.mean([r['survival'] for r in results])
if surv_mean > 100:
    print("\n✓ Vanilla DP baseline 작동. Step 3 (Periodic Phase Encoding)으로 진행 가능.")
elif surv_mean > 30:
    print("\n△ 부분 작동. 진단 셀로 obs distribution 확인 권장.")
else:
    print("\n✗ 즉시 깨짐. 진단 셀 확인 (obs format mismatch 의심).")

## 11. Rollout 진단 — obs 분포 mismatch 체크

In [ ]:
diagnose_obs_distribution(results, data['obs_mean'], data['obs_std'], data['OBS_DIM'])

## 12. 요약

**완료**:
- src 분리 구조 검증
- Vanilla DP 학습/로드 → sampling sanity → Ant rollout 모두 작동

**Drive 산출물**:
- `vanilla_dp_ckpt.pt` (model + EMA + train log)
- `vanilla_dp_loss.png`
- `vanilla_dp_action_dist.png`
- `vanilla_dp_chunk_compare.png`

**다음 단계 (`03_phase_periodic.ipynb`)**:
- src/models.py에 `build_periodic_phase_dp_model` 함수 한 줄 추가 (`global_cond_dim = OH × OD + 2`)
- 새 노트북 첫 셀들은 이 노트북과 거의 동일 (load → build_loaders → build model)
- `cond_fn=periodic_phase_cond_fn` 으로만 변경
- 학습 / sampling / rollout 동일 함수 호출
- Vanilla 대비 어떤 차이가 나는지 비교

이게 src 분리의 진짜 가치 — Step 3 노트북은 이 노트북에서 **두 줄만 바꾸면** 됨.